# ResNet com TensorFlow

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets
import numpy as np
import matplotlib.pyplot as plt

### 1. Carregar e Pré-processar o Conjunto de Dados CIFAR-10

In [ ]:
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Normalizar os valores dos pixels para o intervalo [0, 1]
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0
# Converter rótulos para categorização one-hot
train_labels = tf.keras.utils.to_categorical(train_labels, 10)
test_labels = tf.keras.utils.to_categorical(test_labels, 10)

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i])
    plt.xlabel(class_names[np.argmax(train_labels[i])])
plt.show()

### 2. Definir um Bloco de ResNet

Uma parte fundamental da ResNet é o bloco residual, que permite que a rede aprenda funções de identidade, ajudando a mitigar o problema do gradiente evanescente/explosivo em redes muito profundas.

In [ ]:
def res_block(x, filters, kernel_size=(3, 3), stride=1, conv_shortcut=True, name=None):
    shortcut = x
    x = layers.Conv2D(filters, kernel_size, strides=stride, padding='same', name=f'{name}_conv1')(x)
    x = layers.BatchNormalization(name=f'{name}_bn1')(x)
    x = layers.ReLU(name=f'{name}_relu1')(x)
    x = layers.Conv2D(filters, kernel_size, padding='same', name=f'{name}_conv2')(x)
    x = layers.BatchNormalization(name=f'{name}_bn2')(x)
    if conv_shortcut:
        shortcut = layers.Conv2D(filters, 1, strides=stride, name=f'{name}_shortcut_conv')(shortcut)
        shortcut = layers.BatchNormalization(name=f'{name}_shortcut_bn')(shortcut)
    x = layers.add([x, shortcut], name=f'{name}_add')
    x = layers.ReLU(name=f'{name}_out_relu')(x)
    return x

def build_resnet18(input_shape=(32, 32, 3), num_classes=10):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(64, (3, 3), strides=(1, 1), padding='same', name='conv1')(inputs)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.ReLU(name='relu1')(x)
    x = res_block(x, 64, name='block1_1', conv_shortcut=False) 
    x = res_block(x, 64, name='block1_2', conv_shortcut=False)
    x = res_block(x, 128, stride=2, name='block2_1') 
    x = res_block(x, 128, name='block2_2', conv_shortcut=False)
    x = res_block(x, 256, stride=2, name='block3_1')
    x = res_block(x, 256, name='block3_2', conv_shortcut=False)
    x = res_block(x, 512, stride=2, name='block4_1') 
    x = res_block(x, 512, name='block4_2', conv_shortcut=False)
    x = layers.GlobalAveragePooling2D(name='avg_pool')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    model = models.Model(inputs, outputs, name='resnet18_cifar10')
    return model

model = build_resnet18()
model.summary()

### 3. Compilar e Treinar o Modelo

Definiremos o otimizador, a função de perda e as métricas. Em seguida, treinaremos o modelo com os dados de treinamento.

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

checkpoint_filepath = 'best_resnet_cifar10.weights.h5'
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True)

history = model.fit(train_images, train_labels,
                    epochs=10,
                    validation_data=(test_images, test_labels),
                    batch_size=64,
                    callbacks=[model_checkpoint_callback])

### 4. Avaliar o Modelo

Vamos visualizar o desempenho do modelo no conjunto de teste e plotar o histórico de treinamento.

In [ ]:
model.load_weights(checkpoint_filepath)
loss, accuracy = model.evaluate(test_images, test_labels, verbose=2)
print(f"\nAcurácia do modelo no conjunto de teste: {accuracy:.4f}")
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Acurácia de Treinamento')
plt.plot(history.history['val_accuracy'], label='Acurácia de Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.title('Acurácia de Treinamento vs. Validação')
plt.legend()
plt.grid(True)
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Perda de Treinamento')
plt.plot(history.history['val_loss'], label='Perda de Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.title('Perda de Treinamento vs. Validação')
plt.legend()
plt.grid(True)
plt.show()

## Exemplo de uso de ResNet50 pré-treinada diretamente do `tf.keras.applications`

Quando se utiliza modelos pré-treinados como o ResNet50, geralmente se remove a camada de classificação original (superior) e adiciona-se novas camadas adaptadas ao número de classes do novo conjunto de dados. Além disso, a ResNet50 pré-treinada espera imagens de 224x224 pixels, enquanto o CIFAR-10 possui imagens de 32x32. Portanto, seria necessário um pré-processamento para redimensionar as imagens ou treinar as camadas convolucionais do ResNet50 do zero com o novo `input_shape`.

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Input, Dense, Flatten
from tensorflow.keras.models import Model

#Carregar ResNet50 com pesos pré-treinados (assumindo input_shape=224x224)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
# Congelar as camadas do modelo base para que não sejam treinadas (transfer learning)
for layer in base_model.layers:
    layer.trainable = False
# Adicionar novas camadas para classificação CIFAR-10 (10 classes)
x = base_model.output
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(10, activation='softmax')(x)
model_transfer_learning = Model(inputs=base_model.input, outputs=predictions)
model_transfer_learning.summary()